# Kubeflow Pipelines (ML-Pipeline)

**Ziel:** Das RAG-Ingesting aus `11-hf-dataset-pdf.ipynb` als Kubeflow-Workflow verstehen und ausprobieren.

Dieses Notebook baut eine **lauffähige Demo-Pipeline mit drei Komponenten**:

```text
[PDF einlesen (Mock)] ---> [Chunks erstellen] ---> [Embeddings (Mock)]
      Dataset                  Dataset                 Dataset
```

- **Echt:** KFP-DSL, Pipeline-DAG, Kubernetes-Pods, Datenübergabe als Dataset-Artefakte, Logging, Ausführung über Kubeflow.
- **Simuliert:** PDF-Inhalt und Embeddings. Es werden **keine PDF-Datei**, **kein Hugging-Face-Modell** und **kein LLM** benötigt.
- **Nutzen:** Der Ablauf ist schnell, reproduzierbar und lässt sich in der Kubeflow-UI visualisieren.

Das spätere produktive Pendant würde echte PDF-Dateien einlesen und mit `sentence-transformers` Embeddings für ein HF Dataset erzeugen.


- - -

## 1. Was ist eine Kubeflow ML-Pipeline?

Eine Pipeline beschreibt **welche Aufgaben in welcher Reihenfolge** ausgeführt werden. Das Python-SDK erstellt daraus eine YAML-Pipeline-Spezifikation (KFP IR). Kubeflow führt die Aufgaben als Container in Kubernetes aus.

| Baustein | Bedeutung |
|---|---|
| `@dsl.component` | Python-Funktion, die in einem eigenen Container läuft |
| `@dsl.pipeline` | Verknüpft mehrere Komponenten zu einem gerichteten Graphen (DAG) |
| `Input[Dataset]` / `Output[Dataset]` | Persistierte Artefakte statt Notebook-Variablen |
| `compiler.Compiler()` | Erzeugt die YAML-Datei für die Kubeflow-UI |
| Pipeline Run | Konkrete Ausführung mit Parametern, Status, Logs und Artefakten |

**Wichtige Trennung:** Das Notebook bzw. Python-Skript **definiert** die Pipeline. Erst ein **Run in Kubeflow** führt die Komponenten auf Kubernetes aus. Das Kompilieren führt die Komponenten **nicht** aus.


## 2. Vorbereitung

Für dieses Notebook muss das Kubeflow SDK installiert sein:

In [ ]:
%pip install kfp==2.18.0
from kfp import compiler, dsl
from kfp.dsl import Input, Output, Dataset

print("KFP-SDK-Version:", __import__("kfp").__version__)

### Kubeflow Pipeline

Und [Kubeflow Pipeline](https://www.kubeflow.org/docs/components/pipelines/)

%%bash
export PIPELINE_VERSION=2.18.0
kubectl apply -k "github.com/kubeflow/pipelines/manifests/kustomize/cluster-scoped-resources?ref=$PIPELINE_VERSION"
kubectl wait --for condition=established --timeout=60s crd/applications.app.k8s.io
kubectl apply -k "github.com/kubeflow/pipelines/manifests/kustomize/env/platform-agnostic?ref=$PIPELINE_VERSION"

Das Kubeflow Pipelines Dashboard ist die webbasierte Benutzeroberfläche zur Verwaltung und Überwachung von ML-Pipelines.

- Pipelines: Kompilierte Pipeline-Definitionen (YAML) hochladen und verwalten.
- Runs: Pipelines mit definierten Parametern ausführen und deren Status überwachen.
- Experiments: Mehrere Pipeline-Ausführungen gruppieren und vergleichen.
- Pipeline Graph: Verarbeitungsschritte, Abhängigkeiten und Ausführungsstatus visualisieren.
- Artifacts: Erzeugte Daten, Modelle und weitere Ergebnisse einsehen.
- Logs: Ausgaben und Fehlermeldungen einzelner Pipeline-Schritte analysieren.

In [ ]:
%%bash
kubectl patch svc ml-pipeline-ui -n kubeflow -p '{"spec":{"type":"NodePort"}}'
echo "http://"$(cat ~/data/server-ip)":"$(kubectl get service --namespace kubeflow ml-pipeline-ui -o=jsonpath='{.spec.ports[0].nodePort}')/

## 3. Komponente 1 – PDF einlesen (Mock)

Die Komponente simuliert zwei PDF-Seiten. Im echten Pipeline-Schritt würdest du hier `pypdf.PdfReader` verwenden und das PDF aus einem für den Pod erreichbaren Storage lesen.

`Output[Dataset]` beschreibt ein Artefakt, das Kubeflow nach Abschluss des Schritts im konfigurierten Object Store verwaltet. Der Code schreibt dazu eine JSON-Datei nach `dokument.path`.


In [ ]:
@dsl.component(base_image="python:3.11-slim")
def pdf_lesen_mock(pdf_name: str, dokument: Output[Dataset]):
    """Simuliert die Textextraktion aus einem PDF und erstellt ein Dataset-Artefakt."""
    import json
    from pathlib import Path

    # MOCK: Die Texte werden fest vorgegeben. Es wird KEIN PDF geöffnet.
    seiten = [
        {"page": 1, "text": (
            "Quantum Computing verwendet Qubits anstelle klassischer Bits. "
            "Ein Qubit lässt sich mit einer Superposition beschreiben. "
            "Quantenalgorithmen nutzen Interferenz, um bestimmte Aufgaben zu beschleunigen. "
            "Die Messung liefert ein klassisches Resultat. "
            "Fehlerkorrektur spielt beim Aufbau grosser Quantencomputer eine wichtige Rolle."
        )},
        {"page": 2, "text": (
            "RAG verbindet die Suche nach relevanten Textabschnitten mit einem Sprachmodell. "
            "Bei der Vorbereitung werden Dokumente eingelesen, bereinigt und in Chunks unterteilt. "
            "Ein Embedding repräsentiert normalerweise einen Text als numerischen Vektor. "
            "Der Vektor kann später für eine Ähnlichkeitssuche verwendet werden. "
            "Die Antwortgenerierung gehört zur Inferenz und muss nicht Teil der Batch-Pipeline sein."
        )},
    ]
    eintraege = [
        {"source": pdf_name, "page": seite["page"], "text": seite["text"]}
        for seite in seiten
    ]
    Path(dokument.path).write_text(
        json.dumps(eintraege, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    dokument.metadata["anzahl_seiten"] = len(eintraege)
    dokument.metadata["dateiname"] = pdf_name
    print(f"PDF-Mock: {pdf_name}, {len(eintraege)} Seiten")


## 4. Komponente 2 – Chunks erzeugen

Kubeflow stellt das Ergebnis der ersten Komponente am Pfad `dokument.path` bereit. Die zweite Komponente liest diese Datei ein und schreibt eine neue Dataset-Datei nach `chunks.path`.

`chunk_size` und `overlap` sind in diesem einfachen Mock **Wortzahlen**, nicht Tokenzahlen. Die Parameter erscheinen beim Starten eines Runs in der Kubeflow-UI.


In [ ]:
@dsl.component(base_image="python:3.11-slim")
def chunks_erstellen(
    dokument: Input[Dataset],
    chunks: Output[Dataset],
    chunk_size: int = 25,
    overlap: int = 5,
):
    """Zerlegt das Text-Dataset in überlappende Chunks (hier wortbasiert)."""
    import json
    from pathlib import Path

    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError("Es muss gelten: chunk_size > overlap >= 0")

    seiten = json.loads(Path(dokument.path).read_text(encoding="utf-8"))
    ergebnis = []
    schrittweite = chunk_size - overlap

    for seite in seiten:
        woerter = seite["text"].split()
        for start in range(0, len(woerter), schrittweite):
            ausschnitt = woerter[start : start + chunk_size]
            if not ausschnitt:
                continue
            chunk_index = len(ergebnis)
            ergebnis.append({
                "id": f"chunk-{chunk_index:03d}",
                "text": " ".join(ausschnitt),
                "source": seite["source"],
                "page": seite["page"],
                "chunk": chunk_index,
            })
            if start + chunk_size >= len(woerter):
                break

    Path(chunks.path).write_text(
        json.dumps(ergebnis, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    chunks.metadata["anzahl_chunks"] = len(ergebnis)
    chunks.metadata["chunk_size_woerter"] = chunk_size
    chunks.metadata["overlap_woerter"] = overlap
    print(f"Chunking: {len(seiten)} Seiten -> {len(ergebnis)} Chunks")


## 5. Komponente 3 – Embeddings simulieren

Anstelle eines SentenceTransformer-Modells berechnen wir pro Chunk acht reproduzierbare Zahlen aus einem SHA-256-Hash. **Diese Zahlen sind keine semantischen Embeddings und für Retrieval ungeeignet.**

Das finale Dataset enthält dieselben Kernspalten wie das PDF-RAG-Notebook:
`id`, `text`, `source`, `chunk`, `embedding` (zusätzlich `page`).

Im echten Schritt würdest du `SentenceTransformer.encode(...)` ausführen und das Ergebnis z. B. mit `datasets.Dataset.save_to_disk(...)` speichern. Dafür müsste der Container über die benötigten Python-Pakete und ausreichend CPU/RAM verfügen.


In [ ]:
@dsl.component(base_image="python:3.11-slim")
def embeddings_mock(chunks: Input[Dataset], dataset: Output[Dataset]):
    """Erzeugt deterministische Demo-Vektoren, keine semantischen Embeddings."""
    import hashlib
    import json
    from pathlib import Path

    datensaetze = json.loads(Path(chunks.path).read_text(encoding="utf-8"))
    for eintrag in datensaetze:
        digest = hashlib.sha256(eintrag["text"].encode("utf-8")).digest()
        # ACHTUNG: Hash-basierte Werte sind KEINE echten Text-Embeddings.
        eintrag["embedding"] = [round(wert / 255, 4) for wert in digest[:8]]

    Path(dataset.path).write_text(
        json.dumps(datensaetze, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    dataset.metadata["anzahl_zeilen"] = len(datensaetze)
    dataset.metadata["embedding_dimension"] = 8
    dataset.metadata["hinweis"] = "Nur Demo-Vektoren (SHA-256), nicht fuer Retrieval"
    print(f"Mock-Embeddings: {len(datensaetze)} Zeilen, je 8 Dimensionen")


## 6. Pipeline definieren

Mit `@dsl.pipeline` werden die drei Komponenten verknüpft. Der Output des vorherigen Schritts wird jeweils als Input übergeben.

Es ist **kein manuelles `kubectl create job`** nötig: Kubeflow erstellt die nötigen Kubernetes-Ressourcen beim Pipeline-Run.

Caching wird für die Demo deaktiviert, damit man bei jedem neuen Run alle drei Schritte und ihre Logs sehen kann.


In [ ]:
@dsl.pipeline(
    name="rag-ingest-mock",
    description="Demo: PDF-Mock -> Chunks -> Mock-Embeddings als Dataset-Artefakt",
)
def rag_ingest_mock_pipeline(
    pdf_name: str = "beispiel-quantum.pdf",
    chunk_size: int = 25,
    overlap: int = 5,
):
    dokument = pdf_lesen_mock(pdf_name=pdf_name)
    chunk_task = chunks_erstellen(
        dokument=dokument.outputs["dokument"],
        chunk_size=chunk_size,
        overlap=overlap,
    )
    embed_task = embeddings_mock(chunks=chunk_task.outputs["chunks"])

    # Bei jedem Demo-Run erneut ausführen, damit die Pods und Logs sichtbar werden.
    dokument.set_caching_options(False)
    chunk_task.set_caching_options(False)
    embed_task.set_caching_options(False)


## 7. Pipeline in YAML kompilieren

Die folgende Zelle erzeugt `12-kfp-rag-mock-pipeline.yaml` im **aktuellen Jupyter-Arbeitsverzeichnis**. Dieses YAML ist eine **Kubeflow-Pipeline-Spezifikation**, kein Kubernetes-Deployment.

Wenn du stattdessen das beiliegende Python-Skript verwendest, genügt:

```bash
python 12-kfp-rag-mock.py
```


In [ ]:
from pathlib import Path

PIPELINE_YAML = Path.cwd() / "42-kubeflow-pipeline-rag.yaml"
compiler.Compiler().compile(
    pipeline_func=rag_ingest_mock_pipeline,
    package_path=str(PIPELINE_YAML),
)
print("Pipeline-YAML erstellt:", PIPELINE_YAML)
print("Dateigrösse:", PIPELINE_YAML.stat().st_size, "Bytes")


## 8. In der Kubeflow-UI ausführen

1. `42-kubeflow-pipeline-rag.yaml` downloaden mittels rechter Maustaste
2. Die Kubeflow-UI öffnen und **Pipelines → Upload pipeline** wählen.
3. `42-kubeflow-pipeline-rag.yaml` auswählen und hochladen.
4. **Create run** Standardparameter belassen oder beispielsweise `chunk_size=15`, `overlap=3` wählen und den Run starten.
5. Den **Graphen**, die **Logs** und die **Dataset-Artefakte** der drei Schritte ansehen.

Das Ergebnis ist kein lokal erzeugtes HF Dataset-Verzeichnis: Die JSON-Artefakte werden über die KFP-Artefaktverwaltung im Object Store abgelegt (in deiner Installation SeaweedFS).


In [ ]:
! echo "http://"$(cat ~/data/server-ip)":"$(kubectl get service --namespace kubeflow ml-pipeline-ui -o=jsonpath='{.spec.ports[0].nodePort}')/

- - -
  
## 9. Vom Mock zur echten RAG-Pipeline

| Mock | Spätere Implementierung aus Notebook 11 |
|---|---|
| Fest vorgegebene Seiten | PDF lesen (`pypdf`) aus PVC oder Object Storage |
| Wortbasiertes Chunking | Bereinigter Text und konfigurierbares Chunking |
| SHA-256-Demovektor | Echte Embeddings mit `sentence-transformers` |
| JSON-Dataset-Artefakt | HF Dataset aufbauen und `save_to_disk(...)` auf einem Artefaktpfad |

**Architekturhinweis:** Die Ingest-Pipeline ist ein **Batch-Job**. Interaktive Nutzerfragen, Retrieval und die LLM-Antwort sind üblicherweise ein eigener laufender Dienst, nicht Teil jedes Ingest-Runs.

**Referenzen:** [KFP 2.18 – Python SDK](https://kubeflow-pipelines.readthedocs.io/en/release-2.18/python-sdk.html), [Pipeline kompilieren](https://www.kubeflow.org/docs/components/pipelines/user-guides/core-functions/compile-a-pipeline/), [Pipeline ausführen](https://www.kubeflow.org/docs/components/pipelines/user-guides/core-functions/run-a-pipeline/).
